In [ ]:
import sys
import pandas as pd
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px

   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ----------- ---------------------------- 2.1/7.2 MB 10.7 MB/s eta 0:00:01
   ------------------------ --------------- 4.5/7.2 MB 11.7 MB/s eta 0:00:01
   ------------------------------------ --- 6.6/7.2 MB 10.9 MB/s eta 0:00:01
   ---------------------------------------- 7.2/7.2 MB 9.5 MB/s  0:00:00
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ----------- ---------------------------- 2.9/9.9 MB 15.2 MB/s eta 0:00:01
   ---------------------- ----------------- 5.5/9.9 MB 14.6 MB/s eta 0:00:01
   --------------------------------- ------ 8.4/9.9 MB 14.9 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.9 MB 14.4 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 11.4 MB/s  0:00:00

   --- ------------------------------------  1/11 [Werkzeug]
   ------- --------------------------------  2/11 [retrying]
   ---------- -----------------------------  3/11 [na


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv'
spacex_df = pd.read_csv(URL)
spacex_df.head()

,Unnamed: 0,Flight Number,Launch Site,class,Payload Mass (kg),Booster Version,Booster Version Category
0,0,1,CCAFS LC-40,0,0.0,F9 v1.0 B0003,v1.0
1,1,2,CCAFS LC-40,0,0.0,F9 v1.0 B0004,v1.0
2,2,3,CCAFS LC-40,0,525.0,F9 v1.0 B0005,v1.0
3,3,4,CCAFS LC-40,0,500.0,F9 v1.0 B0006,v1.0
4,4,5,CCAFS LC-40,0,677.0,F9 v1.0 B0007,v1.0


In [5]:
min_payload = 0
max_payload = 10000

app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1('SpaceX Launch Records Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}),

    html.Br(),

    # TASK 1: dropdown
    dcc.Dropdown(
        id='site-dropdown',
        options=[
            {'label': 'All Sites', 'value': 'ALL'},
            {'label': 'CCAFS LC-40', 'value': 'CCAFS LC-40'},
            {'label': 'VAFB SLC-4E', 'value': 'VAFB SLC-4E'},
            {'label': 'KSC LC-39A', 'value': 'KSC LC-39A'},
            {'label': 'CCAFS SLC-40', 'value': 'CCAFS SLC-40'}
        ],
        value='ALL',
        placeholder='Select a Launch Site here',
        searchable=True
    ),

    html.Br(),

    # TASK 2: pie chart output
    html.Div(dcc.Graph(id='success-pie-chart')),

    html.Br(),

    # TASK 3: payload range slider
    html.P('Payload range (Kg):'),
    dcc.RangeSlider(
        id='payload-slider',
        min=0,
        max=10000,
        step=1000,
        marks={0: '0', 2500: '2500', 5000: '5000', 7500: '7500', 10000: '10000'},
        value=[min_payload, max_payload]
    ),

    html.Br(),

    # TASK 4: scatter chart output
    html.Div(dcc.Graph(id='success-payload-scatter-chart')),
])


# TASK 2 callback
@app.callback(
    Output(component_id='success-pie-chart', component_property='figure'),
    Input(component_id='site-dropdown', component_property='value')
)
def get_pie_chart(entered_site):
    if entered_site == 'ALL':
        fig = px.pie(spacex_df[spacex_df['class'] == 1],
                     names='Launch Site',
                     title='Total Success Launches by Site')
    else:
        filtered = spacex_df[spacex_df['Launch Site'] == entered_site]
        counts = filtered['class'].value_counts().reset_index()
        counts.columns = ['class', 'count']
        counts['Outcome'] = counts['class'].map({1: 'Success', 0: 'Failure'})
        fig = px.pie(counts, names='Outcome', values='count',
                     title='Total Launch Outcomes for site ' + entered_site)
    return fig


# ── TASK 4: Callback – Scatter chart ─────────────────────────────────────
@app.callback(
    Output(component_id='success-payload-scatter-chart', component_property='figure'),
    [
        Input(component_id='site-dropdown',   component_property='value'),
        Input(component_id='payload-slider',  component_property='value')
    ]
)
def get_scatter_chart(entered_site, payload_range):
    low, high = payload_range

    # Filter by payload range first
    filtered_df = spacex_df[
        (spacex_df['Payload Mass (kg)'] >= low) &
        (spacex_df['Payload Mass (kg)'] <= high)
    ]

    if entered_site == 'ALL':
        fig = px.scatter(
            filtered_df,
            x='Payload Mass (kg)',
            y='class',
            color='Booster Version Category',
            title='Correlation between Payload and Success for All Sites',
            labels={'class': 'Launch Outcome (0=Fail, 1=Success)'},
            hover_data=['Launch Site']
        )
    else:
        site_df = filtered_df[filtered_df['Launch Site'] == entered_site]
        fig = px.scatter(
            site_df,
            x='Payload Mass (kg)',
            y='class',
            color='Booster Version Category',
            title=f'Correlation between Payload and Success for {entered_site}',
            labels={'class': 'Launch Outcome (0=Fail, 1=Success)'}
        )

    fig.update_yaxes(tickvals=[0, 1], ticktext=['Failure', 'Success'])
    return fig


if __name__ == '__main__':
    app.run(debug=True)

## Insights to Record After Exploring the Dashboard

| # | Question | Answer |
|---|----------|--------|
| 1 | Which site has the **largest** successful launches? | KSC LC-39A |
| 2 | Which site has the **highest success rate**? | KSC LC-39A |
| 3 | Which payload range has the **highest** success rate? | 2k-4k |
| 4 | Which payload range has the **lowest** success rate? | 6k-8k |
| 5 | Which F9 Booster version has the highest success rate? | FT |